In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# 1. Simulate IoT Sensor Data (Temperature, Humidity, Pressure, Vibration)
def generate_iot_data(n_samples=1000):
    np.random.seed(42)
    # Normal operating conditions
    temp = np.random.normal(25, 2, n_samples)
    hum = np.random.normal(50, 5, n_samples)
    press = np.random.normal(1013, 10, n_samples)
    vib = np.random.normal(0.05, 0.01, n_samples)

    data = pd.DataFrame({'temp': temp, 'hum': hum, 'press': press, 'vib': vib})
    return data

# Generate and Preprocess
df = generate_iot_data(2000)
scaler = StandardScaler()
scaled_data = scaler.fit_transform(df)

# Split into training and testing
X_train, X_test = train_test_split(scaled_data, test_size=0.2, random_state=42)

# 2. Build the Autoencoder Model
input_dim = X_train.shape[1]  # 4 features
encoding_dim = 2              # Compress to 2 variables

input_layer = layers.Input(shape=(input_dim,))

# Encoder
encoder = layers.Dense(8, activation='relu')(input_layer)
bottleneck = layers.Dense(encoding_dim, activation='relu')(encoder)

# Decoder
decoder = layers.Dense(8, activation='relu')(bottleneck)
output_layer = layers.Dense(input_dim, activation='sigmoid')(decoder)

autoencoder = models.Model(inputs=input_layer, outputs=output_layer)
autoencoder.compile(optimizer='adam', loss='mse')

# 3. Train the Model
# Note: We use X_train as both input and target
autoencoder.fit(X_train, X_train,
                epochs=50,
                batch_size=32,
                validation_split=0.1,
                verbose=0)

# 4. Anomaly Detection Logic
def detect_anomaly(new_data):
    # Scale the new data
    scaled_new = scaler.transform(new_data)
    # Predict/Reconstruct
    reconstructed = autoencoder.predict(scaled_new)
    # Calculate Mean Squared Error
    mse = np.mean(np.power(scaled_new - reconstructed, 2), axis=1)
    return mse

# Simulate an "Anomaly" (e.g., a sensor overheating)
anomaly_sample = pd.DataFrame({'temp': [85], 'hum': [10], 'press': [900], 'vib': [0.5]})
error = detect_anomaly(anomaly_sample)

print(f"Reconstruction Error for Normal Data: {detect_anomaly(df.head(1))[0]:.4f}")
print(f"Reconstruction Error for Anomaly: {error[0]:.4f}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
Reconstruction Error for Normal Data: 0.5718
Reconstruction Error for Anomaly: 719.3649
